# Bronze Layer: Auto Loader (Incremental Ingestion)

## Purpose
Demonstrate **Auto Loader** for incremental file processing:
* **Streaming ingestion** - Processes new files as they arrive
* **Schema inference** - Automatically detects CSV schema
* **Checkpoint tracking** - Ensures exactly-once processing
* **Serverless-optimized** - `availableNow` trigger for serverless compute

## Data Flow
```
Volume Files → Auto Loader Stream → Bronze Delta Table
/Volumes/.../patients*.csv → healthcare.bronze.patient_raw_auto
```

## Professional Exam Topics
* **Auto Loader vs Batch** - When to use streaming vs batch ingestion
* **Checkpoint management** - Idempotency and exactly-once semantics
* **Schema evolution** - Handling schema changes over time
* **Serverless patterns** - `availableNow` trigger for cost optimization
* **Volume integration** - Unity Catalog Volume as source

## Key Differences from Batch (Exam Focus)
* **Incremental** - Only processes new/changed files
* **Stateful** - Uses checkpoints to track progress
* **Production-grade** - Built-in retry and error handling
* **Cost-efficient** - Processes files as they arrive (no full scans)

In [0]:
%sql
-- Create volumes for Auto Loader metadata
-- _schemas: Stores inferred schema
-- _checkpoints: Tracks processed files for exactly-once semantics

CREATE VOLUME IF NOT EXISTS healthcare.bronze._schemas
COMMENT 'Auto Loader schema inference location';

CREATE VOLUME IF NOT EXISTS healthcare.bronze._checkpoints
COMMENT 'Auto Loader checkpoint location for exactly-once processing';

In [0]:
# Centralized configuration for Auto Loader
from pyspark.sql.functions import current_timestamp, col

# Source configuration
SOURCE_PATH = "/Volumes/healthcare/bronze/raw_files"
FILE_PATTERN = "patients*.csv"  # Only process patient files

# Auto Loader metadata paths
SCHEMA_PATH = "/Volumes/healthcare/bronze/_schemas"
CHECKPOINT_PATH = "/Volumes/healthcare/bronze/_checkpoints/patient_auto_v1"

# Target table
TARGET_TABLE = "healthcare.bronze.patient_raw_auto"

print("="*60)
print("📡 AUTO LOADER: PATIENT INGESTION")
print("="*60)
print(f"\n📂 Source: {SOURCE_PATH}")
print(f"🔍 Pattern: {FILE_PATTERN}")
print(f"📋 Schema: {SCHEMA_PATH}")
print(f"✅ Checkpoint: {CHECKPOINT_PATH}")
print(f"🎯 Target: {TARGET_TABLE}")
print(f"\n✅ Configuration loaded")

In [0]:
# Define Auto Loader streaming DataFrame
# Key options:
# - cloudFiles.format: CSV file type
# - cloudFiles.schemaLocation: Where to persist inferred schema
# - pathGlobFilter: Only process files matching pattern

df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", SCHEMA_PATH)
         .option("pathGlobFilter", FILE_PATTERN)
         .load(SOURCE_PATH)
         .withColumn("source_file", col("_metadata.file_path"))
         .withColumn("ingestion_timestamp", current_timestamp())
)

print("\n✅ Auto Loader stream defined")
print(f"   - Schema inference enabled")
print(f"   - Metadata columns: source_file, ingestion_timestamp")

In [0]:
# Start streaming write
# availableNow=True: Processes all available data then stops (serverless-optimized)
# Alternative: trigger(once=True) for one-time processing

print("\n💾 Starting Auto Loader stream...")

query = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)  # Serverless-optimized trigger
        .toTable(TARGET_TABLE)
)

print(f"\n✅ Stream started successfully")
print(f"   - Stream ID: {query.id}")
print(f"   - Target: {TARGET_TABLE}")
print(f"   - Mode: availableNow (processes available files then stops)")
print(f"   - Checkpoint: {CHECKPOINT_PATH}")

In [0]:
# Comprehensive validation of Auto Loader table
from pyspark.sql.functions import max as spark_max, min as spark_min

verify_df = spark.table(TARGET_TABLE)

print("\n✅ Auto Loader Table Verification:")
print("="*60)

# Record counts
total_records = verify_df.count()
print(f"Total records: {total_records:,}")
print(f"Unique patient IDs: {verify_df.select('Id').distinct().count():,}")

# Ingestion metadata
ingestion_stats = verify_df.agg(
    spark_min("ingestion_timestamp").alias("first_ingestion"),
    spark_max("ingestion_timestamp").alias("last_ingestion")
).collect()[0]

print(f"\n🕒 Ingestion Timeline:")
print(f"  First ingestion: {ingestion_stats['first_ingestion']}")
print(f"  Last ingestion: {ingestion_stats['last_ingestion']}")

# Source files
file_count = verify_df.select("source_file").distinct().count()
print(f"\n📁 Processed files: {file_count}")

# Metadata columns
print(f"\n🏷️  Metadata columns:")
print(f"  - source_file: {'✅' if 'source_file' in verify_df.columns else '❌'}")
print(f"  - ingestion_timestamp: {'✅' if 'ingestion_timestamp' in verify_df.columns else '❌'}")

# Sample data
print("\n📊 Sample data:")
display(verify_df.select("Id", "BIRTHDATE", "FIRST", "LAST", "GENDER", "STATE", "ingestion_timestamp").limit(5))